# Retrain T1.3 smoke: LLM query synthesis on BEIR

Validates T1.3 end-to-end on a configurable BEIR dataset.

**Which dataset to pick**: set `DATASET` at the top of Cell 3.
- `scifact` (~5k docs): BGE-small baseline ~0.72 NDCG@10. High ceiling,
  little room for lift. Use this for smoke only.
- `nfcorpus` (~3.6k docs): BGE-small baseline ~0.32. **Use this to
  actually measure if synth helps**: big headroom, medical/scientific
  domain where generic base models struggle.
- `fiqa` (~57k docs): BGE-small baseline ~0.26. Financial/consumer QA.
  Slower but similar signal to nfcorpus.

Prior runs on this pipeline:
- T1.1 on scifact, prefix only: delta +0.24%.
- T1.2 on scifact, prefix K=5: delta -4.97% (gated out).
- T1.3 on scifact, synth K=1: delta +0.21%.

scifact is ceiling-near for BGE-small. Running on nfcorpus is the
honest test of whether LLM-synthesized queries actually help.

Runtime on T4 with Gemini 2.5 Flash: ~20-35 min (synth dominates).
The JSONL cache makes re-runs over the same corpus free.


In [ ]:
# Cell 1: Setup -- feat/retrain-t13-llm-synth branch.
!pip install -q 'sentence-transformers>=3' torch 'accelerate>=1.1.0' openai
!rm -rf /content/vstash
!git clone --branch feat/retrain-t13-llm-synth https://github.com/stffns/vstash.git /content/vstash
%cd /content/vstash
!pip install -q -e .

In [ ]:
# Cell 2: Configure the LLM backend for query synthesis.
# Three options supported:
#   A) OpenAI / OpenAI-compatible
#   B) Gemini via its OpenAI-compat endpoint (recommended for free tier)
#   C) Ollama running on the Colab VM
#
# Pick ONE. Comment out the other two.

import os
from google.colab import userdata

# === Option A: OpenAI-compatible endpoint ===
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
# SYNTH_BACKEND = "openai"
# SYNTH_MODEL = "gpt-4o-mini"
# OPENAI_BASE_URL = "https://api.openai.com/v1"

# === Option B: Gemini via OpenAI-compat endpoint ===
# Uses Google's official OpenAI-compatible surface:
#   https://ai.google.dev/gemini-api/docs/openai
# Put the key in Colab Secrets as GEMINI_API_KEY.
os.environ["OPENAI_API_KEY"] = userdata.get("GEMINI_API_KEY")
SYNTH_BACKEND = "openai"  # we still go through our openai path
SYNTH_MODEL = "gemini-2.5-flash"  # fast, cheap, more than enough for synthesis
OPENAI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

# === Option C: Ollama on Colab ===
# !curl -fsSL https://ollama.com/install.sh | sh
# import subprocess, time
# ollama_proc = subprocess.Popen(["ollama", "serve"])
# time.sleep(4)
# !ollama pull qwen2.5:3b-instruct
# SYNTH_BACKEND = "ollama"
# SYNTH_MODEL = "qwen2.5:3b-instruct"
# OLLAMA_URL = "http://127.0.0.1:11434"

print(f"synth backend: {SYNTH_BACKEND}  model: {SYNTH_MODEL}")

In [ ]:
# Cell 3: Download BEIR dataset and ingest with base-model embeddings.
import os
import shutil
import sys

# download_beir uses CACHE_DIR = "experiments/data" relative to cwd, and
# that directory is gitignored -- a fresh clone on Colab does not have
# it. Ensure cwd points at the repo root and the cache dir exists
# BEFORE importing the helper module.
os.chdir("/content/vstash")
sys.path.insert(0, "/content/vstash")
os.makedirs("experiments/data", exist_ok=True)

from experiments.beir_benchmark import download_beir, load_beir
from sentence_transformers import SentenceTransformer
from vstash.store import VstashStore

BASE_MODEL = "BAAI/bge-small-en-v1.5"

# ===== Change this to switch corpora =====
# Known baselines for BGE-small-en-v1.5 (NDCG@10 on BEIR test):
#   scifact  ~0.72  (corpus ~5k)      -> high ceiling, low room for lift
#   nfcorpus ~0.32  (corpus ~3.6k)    -> low ceiling, big room for lift
#   fiqa     ~0.26  (corpus ~57k)     -> low ceiling, bigger corpus
DATASET = "nfcorpus"
# ==========================================

STORE_PATH = f"/tmp/retrain_t13_{DATASET}.db"
OUTPUT_PATH = f"/content/retrained_model_t13_{DATASET}"
SYNTH_CACHE = f"/content/retrain_synth_cache_{DATASET}.jsonl"


def path_for_doc(doc_id):
    return f"{DATASET}://{doc_id}"


for p in (
    STORE_PATH,
    STORE_PATH + "-wal",
    STORE_PATH + "-shm",
    OUTPUT_PATH,
    OUTPUT_PATH + ".candidate",
    OUTPUT_PATH + ".old",
):
    if os.path.isdir(p):
        shutil.rmtree(p)
    elif os.path.isfile(p):
        os.remove(p)

cache = download_beir(DATASET)
corpus, queries, qrels = load_beir(cache)
doc_ids = list(corpus.keys())
print(f"[{DATASET}] corpus: {len(doc_ids)} docs | queries: {len(queries)} | qrels: {len(qrels)}")

model = SentenceTransformer(BASE_MODEL)
texts = [(corpus[d].get("title", "") + " " + corpus[d].get("text", "")).strip() for d in doc_ids]
vecs = model.encode(texts, normalize_embeddings=True, show_progress_bar=True, batch_size=128)
print(f"embedded {len(texts)} docs, dim={vecs.shape[1]}")

store = VstashStore(STORE_PATH, embedding_dim=int(vecs.shape[1]))
for doc_id, text, vec in zip(doc_ids, texts, vecs):
    store.add_document(
        path=path_for_doc(doc_id),
        title=corpus[doc_id].get("title", "")[:80] or doc_id,
        chunks=[text],
        embeddings=[list(map(float, vec))],
    )

stats = store.stats()
print(f"vstash store: {stats.documents} docs, {stats.chunks} chunks")

In [ ]:
# Cell 4: Build a VstashConfig pointing at the chosen synth backend.
from vstash.config import VstashConfig

OLLAMA_URL = globals().get("OLLAMA_URL", "http://127.0.0.1:11434")

if SYNTH_BACKEND == "openai":
    cfg = VstashConfig.model_validate(
        {
            "inference": {"backend": "openai"},
            "openai": {
                "api_key": os.environ["OPENAI_API_KEY"],
                "model": SYNTH_MODEL,
                "base_url": OPENAI_BASE_URL,
            },
        }
    )
elif SYNTH_BACKEND == "ollama":
    cfg = VstashConfig.model_validate(
        {
            "inference": {"backend": "ollama"},
            "ollama": {"base_url": OLLAMA_URL, "model": SYNTH_MODEL},
        }
    )
else:
    raise ValueError(f"unknown SYNTH_BACKEND: {SYNTH_BACKEND}")

print(f"cfg.inference.backend = {cfg.inference.backend}")

In [ ]:
# Cell 5: Sanity-check the LLM by synthesizing queries for a handful of chunks.
from vstash.retrain_synth import synthesize_queries

sample_chunks = [{"id": i, "text": texts[i]} for i in range(min(3, len(texts)))]
sample_out = synthesize_queries(
    sample_chunks,
    cfg,
    n_per_chunk=2,
    cache_path=None,  # no caching for the preview
)
for cid, qs in sample_out.items():
    print(f"[chunk {cid}]")
    print("  passage (excerpt):", sample_chunks[cid]["text"][:120], "...")
    for q in qs:
        print("  -", q)

In [ ]:
# Cell 6: Run retrain() with LLM-synthesized queries as the training query source.
import time
from vstash.retrain import qrels_to_eval_queries, retrain as run_retrain

eval_queries = qrels_to_eval_queries(
    queries=queries,
    qrels=qrels,
    path_for_doc_id=path_for_doc,
)
print(f"eval_queries (real qrels): {len(eval_queries)}")

EVAL_NOISE = max(len(doc_ids), 10000)

t0 = time.perf_counter()
result = run_retrain(
    store,
    base_model=BASE_MODEL,
    output_path=OUTPUT_PATH,
    max_queries=1000,  # fewer chunks -> fewer LLM calls -> faster iteration
    epochs=2,
    lr=3e-6,
    batch_size=64,
    eval_queries=eval_queries,
    eval_noise_size=EVAL_NOISE,
    min_gain=0.0,
    synthesize_queries=True,  # T1.3
    synth_n=2,
    synth_cache=SYNTH_CACHE,
    synth_model=SYNTH_MODEL,
    cfg=cfg,
)
elapsed = time.perf_counter() - t0
print(f"\nretrain() finished in {elapsed:.1f}s")
print(f"RetrainResult: {result}")

In [ ]:
# Cell 7: Pretty-printed report.
import json
from pathlib import Path


def fmt_metrics(m):
    if m is None:
        return "    (not computed)"
    return (
        f"    n_queries:  {m.n_queries}\n"
        f"    NDCG@10:    {m.ndcg_at_10:.4f}\n"
        f"    MRR:        {m.mrr:.4f}\n"
        f"    Hit@10:     {m.hit_at_10:.4f}"
    )


print("=" * 60)
print(f"T1.3 end-to-end smoke: {DATASET}, synth queries ({SYNTH_MODEL})")
print("=" * 60)
print(f"n_pairs:       {result.n_pairs}")
print(f"gated_out:     {result.gated_out}")
print(f"min_gain:      {result.min_gain:+.4f}")
print(f"output_path:   {result.output_path}")
print()
print("Baseline:")
print(fmt_metrics(result.baseline))
print()
print("Final:")
print(fmt_metrics(result.final))
print()
if result.baseline is not None and result.final is not None:
    delta = result.delta_ndcg
    print(f"Delta NDCG@10: {delta:+.4f}  ({delta * 100:+.2f}%)")

meta_path = Path(result.output_path or (OUTPUT_PATH + ".candidate")) / "training_meta.json"
if meta_path.exists():
    print()
    print("training_meta.json:")
    print(json.dumps(json.loads(meta_path.read_text()), indent=2))